# Phase 5 & 6: Grad-CAM Explainable AI (XAI) & Confidence Evaluation
## Project: Indian Medicinal Plant Identification using Deep Learning & XAI

This notebook verifies Gradient-weighted Class Activation Mapping (Grad-CAM) to explain the visual features influencing the deep learning predictions.

### XAI Workflow:
$$\\alpha_k^c = \\frac{1}{Z} \\sum_i \\sum_j \\frac{\\partial y^c}{\\partial A_{i,j}^k}$$
$$L_{\\text{Grad-CAM}}^c = \\text{ReLU}\\left(\\sum_k \\alpha_k^c A^k\\right)$$


In [ ]:
import sys, os
sys.path.append(r"d:\mainpojectD7")
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

from src.model import build_model
from src.gradcam import GradCAM, overlay_heatmap
from src.dataset import get_val_test_transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(num_classes=93, backbone_name="mobilenet_v3_large", pretrained=False, checkpoint_path=r"d:\mainpojectD7\saved_models\best_plant_model.pth", device=device)

# Load a test specimen: Tulsi (Ocimum sanctum)
test_dir = r"d:\mainpojectD7\test-20260921T080351Z-1-001\test\95_Ocimum_sanctum"
sample_img_path = os.path.join(test_dir, os.listdir(test_dir)[0])
raw_pil = Image.open(sample_img_path).convert("RGB")

tf = get_val_test_transforms(224)
input_tensor = tf(raw_pil).unsqueeze(0).to(device)

gradcam = GradCAM(model, model.get_gradcam_target_layer())
heatmap, pred_idx, conf = gradcam.generate_heatmap(input_tensor)
gradcam.remove_hooks()

# Generate overlay
raw_resized = np.array(raw_pil.resize((224, 224)))
colored_cam, overlay = overlay_heatmap(raw_resized, heatmap, alpha=0.55)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_resized)
axes[0].set_title("Input Specimen (Tulsi)", fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(colored_cam)
axes[1].set_title("Grad-CAM Heatmap (Jet)", fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title(f"Attribution Overlay (Conf: {conf*100:.1f}%)", fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()
